# PolyPythia seed-replication study

This notebook asks whether a CPS signature is a property of the training regime or an accident of one initialization and data order.

## Design principle

Every seed is measured with the same checkpoint revision, parameter-selection contract, batch construction, projection rank, and phase grid. The seed identifier changes; the instrument does not.

Set `CPS_SEEDS` and `CPS_REVISION` through the Colab environment. The default two-seed run is a harness check, not a population-level conclusion.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

In [ ]:
from cps.notebook import show_environment
runtime = show_environment()

In [ ]:
import dataclasses, os
from cps.notebook import show_config
from cps.pythia.config import load_probe_config

base = load_probe_config("subjects/pythia/configs/polypythia_70m_seed_study.yaml")
seeds = [int(item) for item in os.environ.get("CPS_SEEDS", "1,2").split(",") if item.strip()]
revision = os.environ.get("CPS_REVISION", "step1000")
show_config(base)
print(f"[SEEDS] revision={revision}; seeds={seeds}", flush=True)

## Execute identical probes across seeds

The runner log announces each model repository and writes an independent evidence root. This is important for later grouped cross-validation: no seed should share a manifest or reduced operator with another seed.

In [ ]:
from cps.pythia.runner import run_probe

outputs = []
for index, seed in enumerate(seeds, start=1):
    print(f"\n[SEEDS] ===== seed {seed} ({index}/{len(seeds)}) =====", flush=True)
    model = dataclasses.replace(base.model, run=f"polypythia-70m-seed{seed}", revision=revision)
    output_config = dataclasses.replace(base.output, run_name=f"polypythia-seed-{seed}")
    current = dataclasses.replace(base, model=model, output=output_config)
    outputs.append(run_probe(current))
print("[SEEDS] completed roots:", *outputs, sep="\n  - ", flush=True)

## Compare run-level observables

This table is descriptive. A serious seed study should estimate variance components and test held-out seed prediction rather than compare only maxima.

In [ ]:
import json, pathlib, pandas as pd
from IPython.display import display

rows=[]
for output in outputs:
    root=pathlib.Path(output)
    manifest=json.loads((root/"manifest.json").read_text())
    records=json.loads((root/"couplings.json").read_text())
    rows.append({
        "run": manifest["run_spec"]["name"],
        "revision": manifest["revision"],
        "JVP backend": manifest["jacobian"]["effective_backend"],
        "closure max": manifest["projection"]["maximum_closure_residual"],
        "phase radius max": max(r["metrics"]["spectral_radius_max"] for r in records),
        "transient gain max": max(r["metrics"]["finite_horizon_gain"] for r in records),
    })
seed_frame=pd.DataFrame(rows)
display(seed_frame)
print("[SEEDS] Do not interpret two points as a variance estimate.", flush=True)

In [ ]:
from cps.notebook import export_artifacts
archive = export_artifacts()
print(f"Artifact archive ready for colab-cli download: {archive}", flush=True)